# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook provides a workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not as dict!)
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Identifier: {md.identifier}")
print(f"Published: {md.datePublished}")
print(f"Version: {md.version}")
print(f"License: {md.license}")

## 2. Data Overview
Review available record sets, fields, and their @id values.

In [ ]:
# List all record sets and their @ids
print("Record sets in this dataset:")

record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# For demonstration, get the fields and columns in the first record set
if record_sets:
    main_rs = record_sets[0]
    print(f"\nFields in {main_rs.name} (@id: {main_rs.id}):")
    for fld in main_rs.fields:
        print(f"  - Field: {fld.name}, @id: {fld.id}, dataType: {getattr(fld, 'data_type', None)}")
    # Optionally, show columns if available (for tabular datasets)
    if hasattr(main_rs, 'columns') and main_rs.columns:
        print(f"\nColumns in {main_rs.name} (@id: {main_rs.id}):")
        for col in main_rs.columns:
            print(f"  - Column: {col.name}, @id: {col.id}, dataType: {getattr(col, 'data_type', None)}")

## 3. Data Extraction
Load data from each record set into a DataFrame using their @id values.
Assign `record_set_ids` and always use the @id for selection.

In [ ]:
# Collect all available record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets()]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows from record set: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}\n")

# For reference, display the first few rows (head) for the first record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Preview of data for '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps: filtering, normalization, categorization, grouping, etc.
All field/column references use their `@id`.

In [ ]:
# Pick a numeric field for filtering and normalization
# Here we try to automatically pick the first numeric column (@id), but you can adjust this manually.
main_df = dataframes[main_record_set_id]

# Identify candidate numeric fields (@ids) among columns
sample = main_df.head(1).to_dict(orient='records')[0] if not main_df.empty else {}

def is_number(s):
    try:
        float(s)
        return True
    except:
        return False

numeric_field_id = None
for col in main_df.columns:
    # Test sample row and test if not too obviously categorical
    val = sample.get(col, None)
    if val is not None and is_number(val):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    print(f"Numeric field selected (by @id): {numeric_field_id}")

    # Convert column to numeric (in case dtype is object)
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

    # Set an arbitrary threshold (e.g., 10)
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field in the filtered_df
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to pick a grouping field (@id)
    group_field = None
    # Choose a column with type object and low cardinality, likely categorical
    for col in main_df.columns:
        if col != numeric_field_id and main_df[col].dtype == object:
            n_unique = main_df[col].nunique()
            if 2 <= n_unique <= 8:
                group_field = col
                break
    if group_field is not None:
        print(f"Grouping by field (by @id): {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df)
    else:
        print('No suitable grouping field found.')
else:
    print('No numeric field found for filtering in the main record set.')

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Visualize numeric field distribution
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df is available, plot mean by group
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Cannot visualize: No numeric field found.")

## 6. Conclusion
In this notebook, we demonstrated step-by-step exploration of the FAIR² dataset via `mlcroissant`. We loaded record sets using their Croissant `@id`, listed fields and columns, extracted tabular data, filtered and normalized numeric columns, grouped by categorical attributes, and visualized distributions. 

For more advanced analyses, follow the same pattern, always referencing entities by their `@id` for reproducibility and interoperability.

_Dataset citation_: Liu, Y, Duan, X, Yang, S, Zhang, Y, Han, S 2026 Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer...